# Stuck on Shuffle: demo

This notebook shows how to

1. load the **SABER** benchmark (`NetanelTamir/Stuck-On-Shuffle-SABER`) and look at a few questions,
2. load the **finetuning triplets** (`NetanelTamir/Stuck-On-Shuffle-Finetuning-Dataset`),
3. load one of our finetuned embedding models and compare it with its frozen base on a few multiple-choice questions,
4. compare the two on large-gallery retrieval, with the nearest neighbours of independent and nested queries.

Everything is fetched from the Hugging Face hub on demand. Only the handful of images needed for the
examples is downloaded; the full 9,000-image set is not required. Running the model cells needs an NVIDIA GPU or an Apple silicon Mac
(about 20 GB of GPU or unified memory for the 8B model, 6 GB for a 2B one) and `pip install -r requirements.txt`.

In [1]:
import os, sys

sys.path.insert(
    0, os.path.abspath(".")
)  # repository root, so `stuck_on_shuffle` is importable

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from stuck_on_shuffle import (
    load_saber,
    saber_image,
    load_finetuning_dataset,
    load_model,
    list_models,
    ANSWERS,
    SETTINGS,
)

pd.set_option("display.max_colwidth", 120)

## 1. SABER

Nine CSV files, one per object count (2 to 10), 200 questions each. Every row is a multiple-choice
question: a query and four candidates, each available as an image (`*_img`) and as a caption (`*_txt`).
`ans_1` is the correct candidate (same bindings, pairs re-ordered); `ans_2`, `ans_3`, `ans_4` keep the
same objects and attributes but re-assign them.

In [ ]:
saber = load_saber()  # all nine files concatenated, with an n_objects column
print(saber.shape)
print(saber.n_objects.value_counts().sort_index().to_dict())
saber[["n_objects", "id", "query_txt", "ans_1_txt", "ans_2_txt", "query_img"]].head(3)

Image paths are repo-relative (`images/2_objects/0_query.png`); `saber_image` downloads a single file
and returns a PIL image. Here is the first two-object question in both modalities.

In [ ]:
import textwrap


def show_question(row, figsize=(17, 4.6)):
    cols = ["query"] + ANSWERS
    fig, axes = plt.subplots(1, 5, figsize=figsize)
    for ax, c in zip(axes, cols):
        ax.imshow(saber_image(row[f"{c}_img"]))
        title = (
            "QUERY" if c == "query" else (c + ("  (correct)" if c == "ans_1" else ""))
        )
        ax.set_title(
            title,
            fontsize=11,
            fontweight="bold" if c in ("query", "ans_1") else "normal",
        )
        ax.set_xlabel(textwrap.fill(row[f"{c}_txt"], width=36), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"SABER  |  {row.n_objects} objects  |  id {row.id}", y=1.02)
    plt.tight_layout()
    plt.show()


show_question(saber[saber.n_objects == 2].iloc[0])

Scenes get crowded quickly. A five-object question:

In [ ]:
show_question(saber[saber.n_objects == 5].iloc[0])

## 2. The finetuning triplets

100,000 rows of pure text. Each row has a short `query` naming two or three attribute-object pairs,
a `positive` caption that embeds those pairs in a longer sentence, and two hard negatives that reuse
the *same words* with the attributes (`swapped_attributes`) or the objects (`swapped_objects`)
exchanged. `triplet_type` names the sentence template; the `deferred_bind_*` families were held out
of training and used as a validation probe.

In [ ]:
triplets = load_finetuning_dataset()
print(triplets.shape)
triplets.triplet_type.value_counts()

In [ ]:
for tt in ["pair_swap", "predicative", "triple_bind_cycle", "deferred_bind_respective"]:
    r = triplets[triplets.triplet_type == tt].iloc[0]
    print(f"### {tt}")
    for c in ["query", "positive", "swapped_attributes", "swapped_objects"]:
        print(f"  {c:>18}: {r[c]}")
    print()

## 3. Loading a finetuned model

`load_model` downloads the frozen base model from the hub, attaches our LoRA adapter from
`checkpoints/` (LiNeS scaling already baked in) and returns an object whose `embed_texts` /
`embed_images` give L2-normalised embeddings. Because only the adapter differs from the base model,
`model.without_adapter()` lets us score the *original* model with the same weights in memory.

Pick any of the six released models below. The 8B Qwen3-VL model is the strongest; a 2B model is a
good choice on a smaller GPU.

In [ ]:
print(list_models())
MODEL_NAME = "qwen3vl-embedding-8b"
model = load_model(MODEL_NAME)

### A first look: swapped captions

A binding-blind embedder gives almost the same vector to *"a red mug and a blue kettle"* and
*"a blue mug and a red kettle"*. The finetuned model pushes them apart while keeping a genuinely
different sentence far away.

In [ ]:
texts = [
    "A red mug and a blue kettle.",
    "A blue mug and a red kettle.",  # same words, bindings swapped
    "A blue kettle and a red mug.",  # same bindings, re-ordered  -> should stay close to the first
    "A wooden bench under an oak tree.",
]  # unrelated


def cosine_matrix(model, texts):
    e = model.embed_texts(texts)
    return model.similarity(e, e).numpy()


with model.without_adapter():
    base_sims = cosine_matrix(model, texts)
ours_sims = cosine_matrix(model, texts)


def plot_heatmaps(mats, titles, labels, figsize=(14, 5.6)):
    fig, axes = plt.subplots(
        1, len(mats), figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )
    for k, (ax, m, t) in enumerate(zip(axes, mats, titles)):
        im = ax.imshow(m, cmap="viridis", vmin=0.0, vmax=1.0)
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
        if k == 0:
            ax.set_yticklabels(labels, fontsize=9)  # row labels on the left panel only
        else:
            ax.tick_params(axis="y", left=False)  # shared y axis: no labels, no ticks
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(
                    j,
                    i,
                    f"{m[i, j]:.3f}",
                    ha="center",
                    va="center",
                    fontsize=10,
                    color="white" if m[i, j] < 0.6 else "black",
                )
        ax.set_title(t, fontsize=12)
    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="cosine similarity")
    plt.show()


plot_heatmaps([base_sims, ours_sims], ["base model", "finetuned (ours)"], texts)

## 4. Multiple choice: base vs. finetuned

Each SABER question is scored by comparing the query with its four candidates and picking the most
similar one; `ans_1` is correct. Below are eight questions, two for each modality pairing, spread
over eight object counts: the first question of each object count. The correct candidate has a
green frame and the three wrong ones red frames; the label above each candidate says which model
chose it. Text items are shown as their
caption, image items as the rendered scene.

In [ ]:
from stuck_on_shuffle import saber_image_path

EXAMPLES = {
    "T2T": [2, 6],
    "T2I": [3, 7],
    "I2T": [4, 8],
    "I2I": [5, 10],
}  # pairing -> object counts
GREEN, RED = "#1a9641", "#d7191c"  # correct / wrong candidate frames


def embed_side(model, row, side, cols):
    """(len(cols), d) embeddings of one question's columns, as text ('T') or image ('I')."""
    if side == "T":
        return model.embed_texts([row[f"{c}_txt"] for c in cols])
    return model.embed_images([saber_image_path(row[f"{c}_img"]) for c in cols])


def mcq_pick(model, row, setting):
    q = embed_side(model, row, setting[0], ["query"])
    a = embed_side(model, row, setting[-1], ANSWERS)
    return int((q @ a.T).argmax())


def panel(ax, row, item, side):
    """Draw one item (query or candidate) as an image or as its caption."""
    if side == "I":
        ax.imshow(saber_image(row[f"{item}_img"]))
    else:
        ax.text(
            0.5,
            0.5,
            textwrap.fill(row[f"{item}_txt"], 26),
            ha="center",
            va="center",
            fontsize=11,
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_facecolor("#f4f4f4")
        ax.set_aspect("equal", adjustable="box")  # same square box as the image panels
    ax.set_xticks([])
    ax.set_yticks([])


def show_mcq(setting, row, base_pick, ours_pick, figsize=(17, 4.4)):
    q_side, a_side = setting[0], setting[-1]
    fig, axes = plt.subplots(1, 5, figsize=figsize)
    panel(axes[0], row, "query", q_side)
    axes[0].set_title(
        f"QUERY ({'text' if q_side == 'T' else 'image'})",
        fontsize=11,
        fontweight="bold",
    )
    for k, (ax, a) in enumerate(zip(axes[1:], ANSWERS)):
        panel(ax, row, a, a_side)
        picked = [m for m, p in (("base", base_pick), ("ours", ours_pick)) if p == k]
        ax.set_title(
            a + (f"   <- {' & '.join(picked)}" if picked else ""),
            fontsize=11,
            fontweight="bold" if picked else "normal",
        )
        for s in ax.spines.values():
            s.set_edgecolor(GREEN if k == 0 else RED)
            s.set_linewidth(5)
    fig.suptitle(
        f"{setting}  |  {row.n_objects} objects, id {row.id}  |  "
        f"base picks {ANSWERS[base_pick]}, ours picks {ANSWERS[ours_pick]}",
        y=1.03,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


for setting, counts in EXAMPLES.items():
    for n in counts:
        row = saber[saber.n_objects == n].iloc[0]
        with model.without_adapter():
            base_pick = mcq_pick(model, row, setting)
        show_mcq(setting, row, base_pick, mcq_pick(model, row, setting))

## 5. Large-gallery retrieval: base vs. finetuned

Here the four candidates of every question are pooled into one gallery of 7,200 items and a query
ranks all of it. A gallery item is a positive when it contains every attribute-object binding of the
query (subset containment): independent queries (`id` 0 to 99) have exactly one positive, their own
`ans_1`; nested queries (`id` 100 to 199) also match the larger scenes of their chain.

We embed the whole gallery, captions and images, once with the base model and once with the
finetuned one (this downloads all 9,000 SABER images, about 11 GB, and takes a while on the 8B
model). Then, for eight queries, four independent and four nested with one modality pairing each,
we show the five nearest neighbours under both models. Green frames are positives, red frames
negatives; the title gives each model's average precision for that query.

In [ ]:
from stuck_on_shuffle import (
    saber_queries,
    saber_gallery,
    retrieval_positives,
    mean_average_precision,
    download_saber_images,
)

gallery = saber_gallery(saber)  # all 7,200 candidates, pooled
root = (
    download_saber_images()
)  # their image files (about 11 GB, cached after the first run)
g_paths = [os.path.join(root, p) for p in gallery.img]

QUERIES = [
    ("T2T", 2, 0),
    ("T2I", 3, 1),
    ("I2T", 4, 2),
    ("I2I", 5, 3),  # independent (id < 100)
    ("T2T", 2, 150),
    ("T2I", 3, 120),
    ("I2T", 4, 110),
    ("I2I", 6, 130),
]  # nested (id >= 100)
q_all = saber_queries(saber)
queries = pd.concat(
    [q_all[(q_all.n_objects == n) & (q_all.id == i)] for _, n, i in QUERIES]
).reset_index(drop=True)
positives = retrieval_positives(queries, gallery)  # (8, 7200) bool


def embed_all(model):
    return {
        "gT": model.embed_texts(gallery.txt.tolist(), batch_size=64),
        "gI": model.embed_images(g_paths, batch_size=16),
        "qT": model.embed_texts(queries.txt.tolist()),
        "qI": model.embed_images([saber_image_path(p) for p in queries.img]),
    }


with model.without_adapter():
    E_base = embed_all(model)
E_ours = embed_all(model)


def query_sims(E, qi, setting):
    return (E["q" + setting[0]][qi] @ E["g" + setting[-1]].T).numpy()

In [ ]:
def draw(ax, img_path=None, text=None):
    if img_path is not None:
        ax.imshow(saber_image(img_path))
    else:
        ax.text(
            0.5, 0.5, textwrap.fill(text, 24), ha="center", va="center", fontsize=9.5
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_facecolor("#f4f4f4")
        ax.set_aspect("equal", adjustable="box")  # same square box as the image panels
    ax.set_xticks([])
    ax.set_yticks([])


def show_neighbors(qi, k=5, figsize=(18, 7.6)):
    setting, _, _ = QUERIES[qi]
    q = queries.iloc[qi]
    q_side, g_side = setting[0], setting[-1]
    fig, axes = plt.subplots(
        2, k + 1, figsize=figsize, constrained_layout=True
    )  # keeps row-2 titles clear of row-1 panels
    aps = {}
    for r, (name, E) in enumerate(
        (("base model", E_base), ("finetuned (ours)", E_ours))
    ):
        sims = query_sims(E, qi, setting)
        aps[name] = mean_average_precision(sims[None], positives[[qi]])
        draw(
            axes[r, 0],
            img_path=q.img if q_side == "I" else None,
            text=None if q_side == "I" else q.txt,
        )
        axes[r, 0].set_ylabel(name, fontsize=12, fontweight="bold")
        if r == 0:
            axes[r, 0].set_title(
                f"QUERY ({'text' if q_side == 'T' else 'image'})",
                fontsize=11,
                fontweight="bold",
            )
        for c, j in enumerate(np.argsort(-sims)[:k], start=1):
            g = gallery.iloc[j]
            ax = axes[r, c]
            draw(
                ax,
                img_path=g.img if g_side == "I" else None,
                text=None if g_side == "I" else g.txt,
            )
            ax.set_title(
                f"#{c}  sim {sims[j]:.3f}\n{g.n_objects} obj, id {g.id}, {g.role}",
                fontsize=9,
            )
            for s in ax.spines.values():
                s.set_edgecolor(GREEN if positives[qi, j] else RED)
                s.set_linewidth(5)
    fig.suptitle(
        f"{setting}  |  {queries.subset[qi]} query, {q.n_objects} objects, id {q.id}  |  "
        f"{int(positives[qi].sum())} positive(s) in 7,200  |  AP base {aps['base model']:.2f}, ours {aps['finetuned (ours)']:.2f}",
        fontsize=12,
    )
    plt.show()


for qi in range(len(QUERIES)):
    show_neighbors(qi)

The full protocols over all 1,800 questions and the 7,200-item gallery, for every released model,
are one command each: `scripts/eval_saber_mcq.py --model <name>` and
`scripts/eval_saber_retrieval.py --model <name>`. Both download the 11 GB image set once.